# HDBSCAN — clusters that are not round

K-means failed on the crescents because it insists on spherical clusters of similar size. It
also has two other habits that are often wrong:

- **Every point gets a cluster.** There is no "this is an outlier" option — noise is forced
  into whichever centroid is nearest.
- **You must know `k` in advance.**

**HDBSCAN** fixes all three. It is density-based: a cluster is a region where points are
packed closely together, whatever shape that region happens to be, and points in sparse
regions are labelled **noise** rather than assigned.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import HDBSCAN, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler

RSEED = 42

## Side by side on the shapes that broke k-means

In [ ]:
X_moons, _ = make_moons(n_samples=400, noise=0.06, random_state=RSEED)
X_uneven, _ = make_blobs(n_samples=[400, 60, 60], centers=[[0, 0], [5, 5], [-5, 5]],
                         cluster_std=[2.2, 0.4, 0.4], random_state=RSEED)

datasets = {"two crescents": (X_moons, 2), "unequal sizes": (X_uneven, 3)}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for col, (name, (data, k)) in enumerate(datasets.items()):
    km_labels = KMeans(n_clusters=k, n_init=10, random_state=RSEED).fit_predict(data)
    hdb = HDBSCAN(min_cluster_size=20).fit(data)

    axes[0, col].scatter(data[:, 0], data[:, 1], c=km_labels, cmap="viridis", s=14, alpha=0.75)
    axes[0, col].set_title(f"k-means, k={k} — {name}", fontsize=10)

    noise = hdb.labels_ == -1
    axes[1, col].scatter(data[~noise, 0], data[~noise, 1], c=hdb.labels_[~noise],
                         cmap="viridis", s=14, alpha=0.75)
    axes[1, col].scatter(data[noise, 0], data[noise, 1], c="lightgrey", s=14, marker="x")
    found = len(set(hdb.labels_)) - (1 if noise.any() else 0)
    axes[1, col].set_title(f"HDBSCAN — found {found} clusters, {noise.sum()} noise points", fontsize=10)

plt.tight_layout()
plt.show()

HDBSCAN follows the crescents around their curve, and it was never told how many clusters to
look for. The grey crosses are points it declined to assign.

## `min_cluster_size` — the one parameter that matters

You do not choose the number of clusters. You choose **the smallest group of points you are
willing to call a cluster**, which is usually a much easier question to answer: *would five
customers behaving alike be a segment worth acting on, or noise?*

In [ ]:
for min_size in [5, 15, 30, 60, 120]:
    hdb = HDBSCAN(min_cluster_size=min_size).fit(X_uneven)
    n_clusters = len(set(hdb.labels_)) - (1 if -1 in hdb.labels_ else 0)
    noise = (hdb.labels_ == -1).sum()
    print(f"min_cluster_size={min_size:<5} → {n_clusters} clusters, "
          f"{noise:>3} of {len(X_uneven)} points called noise")

Raise it and small groups get absorbed into noise; lower it and the algorithm finds finer
structure. There is no single right answer — but it is a question about **your problem**
rather than about the geometry, which makes it easier to reason about than `k`.

## On the penguins

In [ ]:
penguins = pd.read_csv("data/penguins.csv").dropna(
    subset=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
)
X = StandardScaler().fit_transform(
    penguins[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]
)

hdb = HDBSCAN(min_cluster_size=20).fit(X)
n_clusters = len(set(hdb.labels_)) - (1 if -1 in hdb.labels_ else 0)

print(f"HDBSCAN found {n_clusters} clusters and called {(hdb.labels_ == -1).sum()} points noise")
print(f"(there are {penguins['species'].nunique()} species in the data)\n")

print(pd.crosstab(penguins["species"], hdb.labels_,
                  rownames=["species"], colnames=["cluster (-1 = noise)"]))

Read the table rather than the cluster count. Whatever HDBSCAN found, the question is
whether it lines up with anything real.

## Choosing between them

| | **K-means** | **HDBSCAN** |
|---|---|---|
| cluster shape | spherical only | any shape |
| number of clusters | you specify `k` | discovered |
| what you tune instead | `k` | `min_cluster_size` |
| outliers | forced into a cluster | labelled noise (`-1`) |
| cluster sizes | prefers similar | happily unequal |
| speed on large data | very fast | slower |
| predicts new points | yes, `.predict()` | not directly |

**Reach for k-means** when you have a reason to expect roughly round, roughly equal groups,
when you need speed, or when you need to assign new points later.

**Reach for HDBSCAN** when the shapes are unknown, when the data has genuine outliers you do
not want absorbed, or when you cannot justify a value of `k`.

> One practical difference worth remembering: k-means gives you `.predict()` for new data
> because a centroid is a portable summary. HDBSCAN's clusters are defined by the density of
> the training points, so there is no equivalent — you would refit.